In [1]:
import sys

print(sys.version)

# LINEAR REGRESSION WAS DONE IN PYTHON 3.12.13

3.10.20 (main, Mar 11 2026, 17:43:48) [Clang 20.1.8 ]


In [23]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

df = pd.read_excel('CLEANED_DATASET_THESIS_FINAL.xlsx') # CHANGE DIRECTORY TO RIGHT PATH

df["TIME_PERIOD"] = pd.to_datetime(df["TIME_PERIOD"].astype(str).str.replace("-M", "-"), format="%Y-%m").dt.to_period("M")

new_rows = []

for country, group in df.groupby('COUNTRY'):
    group = group.sort_values('TIME_PERIOD').copy()

    temp = group[['COUNTRY', 'TIME_PERIOD', 'OBS_VALUE', 'GPR', 'ANNUALIZED_VOLATILITY']].copy()  

    temp['LAG_3'] = temp['OBS_VALUE'].shift(3)
    temp['ROLL_MEAN_3'] = temp['OBS_VALUE'].rolling(3).mean()
    temp['ROLL_MEAN_12'] = temp['OBS_VALUE'].rolling(12).mean()
    temp['LOG_RETURN'] = np.log(temp['OBS_VALUE'] / temp['OBS_VALUE'].shift(1))

    temp['TARGET'] = temp['OBS_VALUE'].shift(-1)
    temp = temp.dropna()

    preds = []
    acts = []

    vol_preds = []
    vol_acts = []

    start_train_size = 24
    window_size = 24

    for i in range(start_train_size, len(temp)):

        train_window = temp.iloc[i-window_size:i].copy()
        
        test_window = temp.iloc[i:i+1].copy()

        features = [
            'OBS_VALUE',
            'GPR',
            'LAG_3',
            'ROLL_MEAN_3',
            'ROLL_MEAN_12',
            'LOG_RETURN']

        X_train = train_window[features]
        y_train = train_window['TARGET']

        X_test = test_window[features]
        y_test = test_window['TARGET']

        model = LinearRegression()
        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)[0]

        preds.append(y_pred)
        acts.append(y_test.values[0])

        obs_value = test_window['OBS_VALUE'].values[0]

        ratio_act = y_test.values[0] / obs_value
        ratio_pred = y_pred / obs_value
        
        if ratio_act <= 0 or ratio_pred <= 0:
            log_ret_act = np.nan
            log_ret_pred = np.nan
        else:
            log_ret_act = np.log(ratio_act)
            log_ret_pred = np.log(ratio_pred)
    
        vol_acts.append(log_ret_act)
        vol_preds.append(log_ret_pred)

    mae_pred = mean_absolute_error(acts, preds)
    mse_pred = mean_squared_error(acts, preds)

    vol_act_series = pd.Series(vol_acts).replace([np.inf, -np.inf], np.nan)
    vol_pred_series = pd.Series(vol_preds).replace([np.inf, -np.inf], np.nan)
    
    vol_act = vol_act_series.rolling(12).std() * np.sqrt(12)
    vol_pred = vol_pred_series.rolling(12).std() * np.sqrt(12)
    
    mask = vol_act.notna() & vol_pred.notna()
    
    mae_vol = mean_absolute_error(vol_act[mask], vol_pred[mask])
    mse_vol = mean_squared_error(vol_act[mask], vol_pred[mask])

    last_row = temp.iloc[-1]

    X_next = last_row[features].to_frame().T
    prediction = model.predict(X_next)[0]

    new_time = group['TIME_PERIOD'].iloc[-1] + 1

    prev_value = last_row['TARGET']
    per_change = (prediction - prev_value) / prev_value

    log_return = np.log(prediction / prev_value)

    last_11 = temp['LOG_RETURN'].dropna().iloc[-11:]
    last_12 = pd.concat([last_11, pd.Series([log_return])])
    st_dev = last_12.std()

    annualized_vol = st_dev * np.sqrt(12)

    new_row = {
        'COUNTRY': country,
        'INDICATOR': group['INDICATOR'].iloc[0],
        'TYPE_OF_TRANSFORMATION': group['TYPE_OF_TRANSFORMATION'].iloc[0],
        'FREQUENCY': group['FREQUENCY'].iloc[0],
        'TIME_PERIOD': new_time,
        'PRED_VAL_LINREG': prediction,
        'PER_CHANGE': per_change,
        'LOG_RETURN': log_return,
        'ST_DEV': st_dev,
        'ANNUALIZED_VOLATILITY': annualized_vol,
        'LINREG_PRED_MAE': mae_pred, 
        'LINREG_PRED_MSE': mse_pred,
        'LINREG_VOL_MAE': mae_vol,
        'LINREG_VOL_MSE': mse_vol
    }

    new_rows.append(new_row)

df_new = pd.DataFrame(new_rows)    
df_updated = pd.concat([df, df_new], ignore_index=True)   
df_updated = df_updated.sort_values(['COUNTRY', 'TIME_PERIOD'])   
df_updated.to_excel('LINREG_RW_Updated_Clean_DF.xlsx', index=False)


In [25]:
df_metrics = pd.read_excel('LINREG_RW_Updated_Clean_DF.xlsx')

print(df_metrics['LINREG_PRED_MAE'].describe())

print(df_metrics['LINREG_PRED_MSE'].describe())

print(df_metrics['LINREG_VOL_MAE'].describe())

print(df_metrics['LINREG_VOL_MSE'].describe())

count    110.000000
mean       9.639706
std       48.531178
min        0.006944
25%        0.037211
50%        0.124303
75%        0.872289
max      438.212407
Name: LINREG_PRED_MAE, dtype: float64
count       110.000000
mean       4553.913479
std       34003.518009
min           0.000160
25%           0.003405
50%           0.030835
75%           2.461038
max      343309.588815
Name: LINREG_PRED_MSE, dtype: float64
count    110.000000
mean       0.031240
std        0.012925
min        0.002659
25%        0.026138
50%        0.027340
75%        0.033241
max        0.089987
Name: LINREG_VOL_MAE, dtype: float64
count    110.000000
mean       0.002916
std        0.007341
min        0.000017
25%        0.001099
50%        0.001330
75%        0.001945
max        0.071239
Name: LINREG_VOL_MSE, dtype: float64
